# Corpus CSV To Layer A Shelves Smoke Pipeline

This notebook walks through the implemented FoodScholar pipeline end to end, starting from the current CSV corpus format and ending with Layer A FoodOn shelves in the graph store.

Flow: CSV chunks -> normalized `Chunk` objects -> optional real-corpus streaming -> Parquet round-trip -> in-memory chunk storage -> external annotation merge -> mini FoodOn ontology -> offline annotation -> Layer A shelf build -> graph inspection -> shell validation.

## Setup

In [ ]:
import os
import sys
from collections import Counter
from itertools import islice
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

FIXTURE_CSV = REPO_ROOT / "tests" / "fixtures" / "corpus_chunks.csv"
MINI_FOODON = REPO_ROOT / "tests" / "fixtures" / "mini_foodon.obo"
REAL_CORPUS = Path("/mnt/workspaces/foodscholar/corpus/big")
OUT_PARQUET = Path("/tmp/foodscholar_chunks_smoke.parquet")
PYTHON_BIN = os.environ.get("PYTHON_BIN", "/mnt/miniconda3/envs/foodscholar/bin/python")

from foodscholar import FoodOnAPI, FoodScholar
from foodscholar.corpus import ChunkAnnotation, iter_chunks, load_chunks, merge_annotations
from foodscholar.corpus import write_chunks_parquet
from foodscholar.io.graph import Card
from foodscholar.layer_a import shelf_id_for_foodon
from foodscholar.ontology import load_ontology

print("Repository root:", REPO_ROOT)
print("Fixture CSV used for the deterministic walkthrough:", FIXTURE_CSV)
print("Python executable used by shell validation cells:", PYTHON_BIN)

## 1. Read CSV Corpus Chunks

In [ ]:
# Load the current corpus CSV shape: chunk_id, chunk_text, type, chunk_metadata.
# The loader converts each row into the package-level Chunk model.
chunks = load_chunks(FIXTURE_CSV)

print("Step 1 - CSV fixture loaded and normalized into Chunk objects")
print("Rows loaded:", len(chunks))
for chunk in chunks:
    print(f"- chunk_id={chunk.chunk_id}")
    print("  source:", chunk.source_type, "section:", chunk.section_type, "doc:", chunk.source_doc_id)
    print("  text preview:", chunk.text[:90])
    print("  preserved source_metadata:", chunk.source_metadata)

## 2. Stream The Real Corpus Directory

In [ ]:
# This proves the same loader can stream the real corpus directory without loading it all into memory.
print("Step 2 - Streaming corpus source:", REAL_CORPUS)
if REAL_CORPUS.exists():
    count = 0
    by_type = Counter()
    first = None
    for chunk in iter_chunks(REAL_CORPUS):
        count += 1
        by_type[chunk.source_type] += 1
        if first is None:
            first = chunk
    print("Total streamed chunks:", count)
    print("Chunks by source_type:", dict(by_type))
    print("First streamed chunk:", first.chunk_id, first.source_type, first.source_doc_id)
else:
    print("Real corpus not found; continuing with fixture only:", REAL_CORPUS)

## 3. Normalize To Parquet And Read Back

In [ ]:
# Write a normalized sample to Parquet and read it back.
# This checks the intermediate storage format preserves source metadata.
source = REAL_CORPUS if REAL_CORPUS.exists() else FIXTURE_CSV
sample = list(islice(iter_chunks(source), 1000))
n_written = write_chunks_parquet(sample, OUT_PARQUET)
restored = load_chunks(OUT_PARQUET)

assert len(restored) == n_written
assert restored[0].source_metadata == sample[0].source_metadata
print("Step 3 - Normalized Parquet round-trip")
print("Input source:", source)
print("Sample chunks written:", n_written)
print("Parquet output:", OUT_PARQUET)
print("Round-tripped chunks loaded:", len(restored))
print("First restored chunk_id:", restored[0].chunk_id)
print("First restored source_metadata:", restored[0].source_metadata)

## 4. Store Normalized Chunks In FoodScholar

In [ ]:
# Store normalized chunks in the FoodScholar facade using the in-memory backend.
# The same ChunkStore protocol will be implemented by Elasticsearch later.
fs = FoodScholar.in_memory()
fs.upsert_chunks(restored)

batch_sizes = [len(batch) for batch in fs.chunk_store.iter_chunks(batch_size=250)]
print("Step 4 - In-memory FoodScholar chunk storage")
print("Stored chunks in fs.chunk_store:", len(fs.chunk_store.scan()))
print("iter_chunks(batch_size=250) produced batch sizes:", batch_sizes)
print("Total batches:", len(batch_sizes))

## 5. Merge External Annotation Output By Chunk ID

In [ ]:
# Simulate the external extraction/linking team output.
# The merge is keyed by chunk_id and updates only annotation fields.
first_id = restored[0].chunk_id
fake_foodon_id = "TEST:0000008"  # olive oil in tests/fixtures/mini_foodon.obo
updated = merge_annotations(
    fs.chunk_store,
    [ChunkAnnotation(chunk_id=first_id, foodon_ids=[fake_foodon_id, fake_foodon_id])],
)
assert updated == 1
assert fs.chunk_store.get(first_id).foodon_ids == [fake_foodon_id]

print("Step 5 - External annotation merge")
print("Updated chunk count:", updated)
print("Updated chunk_id:", first_id)
print("Stored foodon_ids after de-duplication:", fs.chunk_store.get(first_id).foodon_ids)

## 6. Load Mini FoodOn And Run Offline Annotation

In [ ]:
# Run the existing offline annotation path on the small fixture corpus.
# This uses KeywordNER + ThreeTierLinker over the mini FoodOn fixture.
fs_annotate = FoodScholar.in_memory()
fs_annotate.attach_ontology(FoodOnAPI(load_ontology(MINI_FOODON), prefix_filter=None))
fs_annotate.upsert_chunks(chunks)
meta = fs_annotate.annotate()

print("Step 6 - Offline annotation against mini FoodOn")
print("Annotate artifact_id:", meta.artifact_id)
print("Chunks annotated:", meta.record_count)
for chunk in fs_annotate.chunk_store.scan():
    print(f"- chunk_id={chunk.chunk_id}")
    print("  mentions:", [m.text for m in chunk.mentions])
    print("  linked foodon_ids:", chunk.foodon_ids)

## 7. Build Layer A Shelves From Stored FoodOn IDs

In [ ]:
# Build Layer A from the foodon_ids already stored on chunks.
# min_support=1 keeps the tiny fixture visible; production defaults are stricter.
fs_layer_a = fs_annotate
fs_layer_a.config.layer_a.min_support = 1
layer_a_meta = fs_layer_a.build_layer_a()
olive_shelf_id = shelf_id_for_foodon("TEST:0000008")

assert fs_layer_a.graph_store.get_shelf(olive_shelf_id) is not None
print("Step 7 - Layer A shelf build")
print("Layer A artifact_id:", layer_a_meta.artifact_id)
print("Shelves written to graph_store:", layer_a_meta.record_count)
print("Shelf id derived from TEST:0000008:", olive_shelf_id)
print("Resolved olive oil shelf:", fs_layer_a.graph_store.get_shelf(olive_shelf_id))

print("Generated shelves, ordered by depth:")
for shelf in sorted(fs_layer_a.graph_store.list_shelves(), key=lambda s: (s.depth, s.label)):
    print(
        f"- {shelf.shelf_id} | label={shelf.label} | depth={shelf.depth} | "
        f"parent={shelf.parent_shelf_id} | supported_chunks={shelf.chunk_count}"
    )

## 8. Inspect Layer A Shelves Through Graph Handles

In [ ]:
# The formal attach phase is still deferred.
# For the smoke walkthrough, attach one fixture chunk manually through GraphView.
fs_graph = fs_layer_a
fs_graph.graph.attach_chunks(["c-abstract"], shelf=olive_shelf_id)
fs_graph.graph.add_theme(
    theme_id="t-heart",
    label="Cardiovascular outcomes",
    shelf_ids=[olive_shelf_id],
    discovered_by="leiden",
    discovery_version="notebook-smoke",
)
fs_graph.graph.attach_chunks(["c-abstract"], theme="t-heart")
fs_graph.graph.add_card(Card(
    card_id="card-olive",
    target_id=olive_shelf_id,
    target_type="shelf",
    title="Olive oil",
    summary="Fixture card for the smoke notebook.",
    evidence_quality="medium",
    cited_chunk_ids=["c-abstract"],
    llm_model=fs_graph.llm.model_id,
    prompt_version="notebook-smoke",
))

shelf = fs_graph.graph.shelf(olive_shelf_id)
print("Step 8 - Graph handles over generated Layer A shelves")
print("Graph summary:", fs_graph.graph.summary())
print("Current shelf:", shelf.shelf_id, shelf.label)
print("Chunks attached to olive oil shelf:", [c.chunk_id for c in shelf.chunks()])
print("Themes attached to olive oil shelf:", [t.theme_id for t in shelf.themes()])
print("Card cited chunks:", [c.chunk_id for c in shelf.card().cited_chunks()])
print("Scoped search for 'olive oil' inside the shelf:", [c.chunk_id for c in fs_graph.graph.search("olive oil", shelf=olive_shelf_id, k=5)])

## 9. Shell Validation - Focused Layer A Tests

In [ ]:
# This shell cell verifies the implemented contracts directly through pytest.
print("Step 9 - Running focused unit tests for storage annotations and Layer A shelf building")
print("Command: pytest tests/unit/test_layer_a.py tests/unit/test_annotation_merge.py tests/unit/test_in_memory_stores.py -q")
!cd "{REPO_ROOT}" && PYTHONPATH="{SRC}" "{PYTHON_BIN}" -m pytest tests/unit/test_layer_a.py tests/unit/test_annotation_merge.py tests/unit/test_in_memory_stores.py -q

## 10. Shell Validation - Corpus Fixture To Layer A

In [ ]:
# This shell cell runs the end-to-end fixture smoke script.
# The script performs: CSV ingestion -> Parquet round-trip -> in-memory storage -> fake annotation merge -> Layer A shelf build.
print("Step 10 - Running fixture corpus smoke through Layer A")
print("Command: scripts/smoke_corpus_storage.sh with CORPUS_DIR=tests/fixtures/corpus_chunks.csv and SAMPLE_SIZE=3")
!cd "{REPO_ROOT}" && CORPUS_DIR="{FIXTURE_CSV}" SAMPLE_SIZE=3 OUT_PARQUET="/tmp/foodscholar_notebook_layer_a_shell.parquet" PYTHONPATH="{SRC}" PYTHON_BIN="{PYTHON_BIN}" bash scripts/smoke_corpus_storage.sh